# Skill 5 · Day 5 牛津 Tutorial LLM 仿真 (v6.0 学习科学层)

## Cell 1 · Persona Prompt (Oxford + HBS + Hattie)

> **Persona**: You are an Oxford tutorial fellow in **Agent 生产部署 (Production deployment: vLLM serving, LangSmith tracing, observability, scaling, cost control)**. You tutor one student at a time in a 50-minute tutorial.
>
> **Rules of engagement**:
> 1. **Never give direct answers.** Use Socratic questioning to make the student reason through the五大挑战 (可靠性/成本/延迟/可观测性/安全合规) and the trade-offs of LangSmith `@traceable` / tiktoken BPE / ResilientLLM fallback / CI/CD 评估门禁 / vLLM PagedAttention.
> 2. **Act as HBS devil's advocate.** When the student proposes a deployment design, steelman the opposite (e.g. "why not just call the commercial API forever instead of self-hosting vLLM?").
> 3. **Reject vague claims.** If the student says "LangSmith makes things observable" without naming which span fields (input/output/latency_ms/tokens/tool_name) or which of the three monitoring tiers it belongs to, push back.
> 4. **End each turn with a probing question.** Use the five Socratic probes: 为什么 (why) / 反例 (counter-example) / 若前提变 (what if premise changes) / 凭什么 (on what basis) / 如何 (how).
> 5. **Hattie 4-level feedback** at the close of the tutorial (see Cell 5): [TASK] / [PROCESS] / [SELF-REG] / [FEED-FORWARD]. Avoid Self-level praise; focus on task, process, and self-regulation.

> **Anti-stall note**: This notebook uses **static if/else branches** to simulate Socratic follow-up. No real openai/anthropic API call is made. The student's free-text answer is matched against expected concept keywords (LangSmith / tiktoken / cl100k_base / RedisSemanticCache / RateLimitError / PagedAttention / Mixtral 8x7B / etc.) and the corresponding Socratic probe is returned.

## Cell 2 · Pre-Tutorial Task (Forced Retrieval, 不可跳过)

牛津 tutorial 的前置门槛: 学生须在 tutorial 前独立完成一段 retrieval, 否则 tutorial 无意义。请在运行 Cell 3 前, 先在自己的草稿本上写完以下两段 (各 >=150 字), 写完再进入 Cell 3:

### Pre-task A (可观测性 + 成本)
用你自己的话解释: 为什么 PoC 阶段"能跑起来就行"的思维是生产化的最大敌人? 如果你给一个营销 Agent (检索知识库 -> 生成小红书文案 -> 工具改写) 加 LangSmith trace, 你会在哪 3 个位置加 `@traceable`? 每个 span 应记录哪 4 个字段? 用 tiktoken 算 token 时为什么必须用 `cl100k_base` 而不是 `len(text)/4`?

### Pre-task B (灾备 + CI/CD + 前沿)
画出 ResilientLLM 的 4 级 fallback 链 (主模型 -> 备用 -> 缓存 -> 默认模板), 标出每级 try/except 应捕获的特定异常。CI/CD 评估门禁的 3 个阈值各是多少 (完成率/幻觉率/安全违规率)? vLLM 的 PagedAttention 与投机解码、MoE 各自的降本机制是什么? 三者能否叠加?

> 写完后再运行下一 cell。Tutorial fellow 会在 Socratic 追问中检验你的草稿是否经得起反诘。

In [ ]:
# Cell 3 · Multi-turn Socratic Loop (static simulation, >=4 turns, >=5 Socratic probes)
# 本 cell 不调用任何真实 LLM API; 用 if/else 关键词匹配模拟 Socratic 追问。
import json, os, time

STUDENT_MODEL_PATH = "student_model.json"

# 5 个苏格拉底探针 (为什么/反例/若前提变/凭什么/如何), 散布在 4+ 轮中
PROBES = {
    "why":        "为什么你认为这个位置必须加 @traceable? 不加会丢失哪个字段 (input/output/latency_ms/tokens/tool_name)?",
    "counter":    "给个反例: 如果 Agent 只有 2 步 (无工具调用) 你这个设计还成立吗? P95 瓶颈会落在哪一步?",
    "premise":    "若前提变了 -- 主模型 gpt-4o 突然停服 (不是限流), 你的 fallback 链第一级 try/except 还能捕获 RateLimitError 吗? 应该捕获哪个异常?",
    "basis":      "凭什么用 cl100k_base 而不是 p50k_base? GPT-4o 系列到底用哪个编码? 你怎么验证?",
    "how":        "如何用 ThreadPoolExecutor 模拟 50 并发? 每个线程独立调用 Agent 还是共享一个 ResilientLLM 实例? 共享实例会不会有线程安全问题?",
}

# 静态 keyword -> Socratic 响应映射 (4 轮, 每轮一个主题)
ROUND_FLOW = [
    {
        "topic": "可观测性 - LangSmith trace 配置",
        "expect_keywords": ["@traceable", "wrap_openai", "span", "tokens", "latency", "三层"],
        "probe_key": "why",
        "success_followup": "你提到了 @traceable。好, 那么下一步: wrap_openai 自动 instrument 了 OpenAI 调用, 但如果你的 Agent 还调了一个非 OpenAI 的工具 (比如本地检索器), 这个工具调用会被 wrap_openai 抓到吗? 不会的话你怎么办?",
    },
    {
        "topic": "成本 - tiktoken BPE 计 token",
        "expect_keywords": ["tiktoken", "cl100k_base", "BPE", "定价", "$/M", "日均万次"],
        "probe_key": "basis",
        "success_followup": "你说对了 cl100k_base。现在反诘: 同一段中文 prompt 用 cl100k_base 计出来是 60 token, 用 len(text)/4 估出来是 30, 差了一倍。这个差异在日均万次请求下月成本差多少美元? 你怎么向 CFO 解释这个差异?",
    },
    {
        "topic": "灾备 - ResilientLLM 4 级 fallback",
        "expect_keywords": ["fallback", "gpt-4o-mini", "RedisSemanticCache", "默认模板", "RateLimitError", "try/except"],
        "probe_key": "premise",
        "success_followup": "你的 fallback 链设计得不错。现在 HBS devil's advocate: 假设 Redis 自己挂了 (缓存层不可用), 你的 4 级链会跳过缓存直接到默认模板吗? 默认模板的营销文案质量够不够上线? 你怎么量化'够不够'?",
    },
    {
        "topic": "前沿 - vLLM / 投机解码 / MoE",
        "expect_keywords": ["vLLM", "PagedAttention", "KV Cache", "continuous batching", "投机解码", "draft model", "MoE", "Mixtral", "2/8 专家"],
        "probe_key": "counter",
        "success_followup": "你区分了投机解码 (降延迟) 与 MoE (降单次计算量)。最后追问: 自建 vLLM 需要固定 GPU 月租, 商业 API 是变动 token 成本。在日均万次请求下, 月请求数到哪个临界点, 自建 vLLM 才比商业 API 便宜? 你怎么算这个 break-even?",
    },
]

def load_student_model():
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"turn": 0, "mastery": {}, "blindspots": [], "completed_rounds": []}

def save_student_model(sm):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

def socratic_respond(student_answer, round_idx, sm):
    """静态 if/else 模拟 Socratic 追问 - 不调 API。"""
    round_def = ROUND_FLOW[round_idx]
    topic = round_def["topic"]
    expects = round_def["expect_keywords"]
    probe_key = round_def["probe_key"]
    answer_lower = student_answer.lower().replace("_", "").replace("-", "")

    hit = [k for k in expects if k.lower().replace("_", "").replace("-", "") in answer_lower]
    hit_count = len(hit)
    coverage = hit_count / len(expects)
    sm["mastery"][topic] = round(coverage, 2)

    if coverage >= 0.6:
        msg = f"[Round {round_idx+1}] 主题: {topic}\n"
        msg += f"  命中关键词 {hit_count}/{len(expects)}: {hit}\n"
        msg += f"  覆盖率 {coverage:.0%} - 暂时过关, 但不要满足。\n"
        msg += f"  Socratic 探针 ({probe_key}): {PROBES[probe_key]}\n"
        msg += f"  Fellow 追问: {round_def['success_followup']}\n"
        if coverage < 1.0:
            missed = [k for k in expects if k not in hit]
            msg += f"  你漏了这些关键词, 复习时重点看: {missed}\n"
            sm["blindspots"].append({"topic": topic, "missed": missed, "round": round_idx})
    else:
        msg = f"[Round {round_idx+1}] 主题: {topic}\n"
        msg += f"  命中关键词 {hit_count}/{len(expects)}: {hit}\n"
        msg += f"  覆盖率 {coverage:.0%} - 不达标, 拒绝你的模糊陈述。\n"
        msg += f"  Oxford fellow 反诘: 你提到的概念太泛, 没有命中本单元的核心术语。回去读 notes.md 关键回顾 + starter.ipynb 对应 TODO, 再来回答。\n"
        msg += f"  Socratic 探针 ({probe_key}): {PROBES[probe_key]}\n"
        sm["blindspots"].append({"topic": topic, "missed": [k for k in expects if k not in hit], "round": round_idx})
        sm["blindspots"].append({"topic": topic, "missed": ["整轮覆盖率不足"], "round": round_idx})

    sm["turn"] += 1
    return msg

# ---- 主循环: 4 轮 Socratic tutorial ----
sm = load_student_model()
print("=" * 70)
print("牛津 Tutorial 仿真开始 (4 轮, 静态模拟, 不调真实 LLM API)")
print("=" * 70)

# 模拟学生的 4 轮回答 (实际使用时替换为真实学生输入)
# 用 4 个不同质量的样本答案演示 if/else 分支
SAMPLE_ANSWERS = [
    # Round 1 答案 (合格)
    "我会在知识库检索、LLM 生成文案、工具改写三步都加 @traceable 装饰器, "
    "用 wrap_openai 自动 instrument OpenAI 调用, 每个 span 记录 input/output/latency_ms/tokens/tool_name, "
    "对应三层监控的第二层 APM。",
    # Round 2 答案 (部分合格, 命中 cl100k_base 但漏了日均万次)
    "用 tiktoken 的 cl100k_base 编码 (因为 GPT-4o 系列用它), 是 BPE 分词器, "
    "结合定价 $/M token 算单次成本, 月成本再乘 30。",
    # Round 3 答案 (不合格 - 模糊)
    "fallback 就是主模型挂了换备用, 备用挂了换缓存, 缓存挂了换模板。",
    # Round 4 答案 (合格 - 前沿)
    "vLLM 用 PagedAttention 优化 KV Cache 内存管理, 支持 continuous batching, "
    "吞吐 14-24x; 投机解码用 draft model 候选 + 大模型验证降延迟 2-3x; "
    "MoE 如 Mixtral 8x7B 只激活 2/8 专家降单次计算量。三者可叠加。",
]

for i, ans in enumerate(SAMPLE_ANSWERS):
    print(f"\n--- Turn {i+1}/4 ---")
    print(f"学生回答: {ans[:80]}...")
    resp = socratic_respond(ans, i, sm)
    print(resp)
    time.sleep(0.05)

sm["completed_rounds"] = [i+1 for i in range(len(SAMPLE_ANSWERS))]
save_student_model(sm)
print("\n" + "=" * 70)
print(f"Tutorial 结束。已完成 {len(SAMPLE_ANSWERS)} 轮。student_model.json 已更新。")
print(f"盲点数: {len(sm['blindspots'])}。下一步见 Cell 4 / Cell 5。")
print("=" * 70)


In [ ]:
# Cell 4 · student_model.json 读写 (记录掌握度 / 盲点)
# 与 Cell 3 共用同一文件; 这里独立展示结构与读写 API, 供 alignment.md 的 AT 评估取数。
import json, os

STUDENT_MODEL_PATH = "student_model.json"

def read_student_model():
    if not os.path.exists(STUDENT_MODEL_PATH):
        return {"turn": 0, "mastery": {}, "blindspots": [], "completed_rounds": []}
    with open(STUDENT_MODEL_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

def write_student_model(sm):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(sm, f, ensure_ascii=False, indent=2)

def summarize(sm):
    print("=== student_model.json 摘要 ===")
    print(f"已完成轮数: {len(sm.get('completed_rounds', []))}")
    print(f"总盲点数: {len(sm.get('blindspots', []))}")
    print("\n各主题掌握度 (mastery, 0-1):")
    for topic, score in sm.get("mastery", {}).items():
        bar = "#" * int(score * 20) + "." * (20 - int(score * 20))
        flag = " [PASS]" if score >= 0.6 else " [WEAK]"
        print(f"  [{bar}] {score:.0%} {topic}{flag}")
    print("\n盲点清单 (供 alignment.md AT 取数 + Cell 5 反馈):")
    for b in sm.get("blindspots", []):
        print(f"  - Round {b.get('round', '?')+1} | {b['topic']} | 漏: {b.get('missed', [])}")

sm = read_student_model()
summarize(sm)

# 教师侧: 根据 mastery 决定是否触发 practice.md Weak Loop
weak_topics = [t for t, s in sm.get("mastery", {}).items() if s < 0.6]
if weak_topics:
    print(f"\n[触发 practice.md Weak Loop] 连续未达 0.6 的主题: {weak_topics}")
    print("  -> 回退到该主题 drill 的 Faded 阶段 + 补 Worked example + 复述决策点")
else:
    print("\n[全部达标] 进入 alignment.md 对应 AT 评估任务")


## Cell 5 · Hattie 4 级 Formative Feedback

> 依据 Hattie & Timperley (2007) "The Power of Feedback"。本 cell 基于 Cell 4 的 student_model.json 生成 4 级反馈。**避免 Self 级表扬** (Hattie 实证: Self 级表扬对学习效果几乎为零, d=0.04), 聚焦 Task / Process / Self-Reg / Feed-Forward。


In [ ]:
# Cell 5 (code) · 基于 student_model.json 生成 Hattie 4 级反馈
import json, os

with open("student_model.json", "r", encoding="utf-8") as f:
    sm = json.load(f)

mastery = sm.get("mastery", {})
blindspots = sm.get("blindspots", [])
turn = sm.get("turn", 0)

# --- [TASK] 任务级反馈: 这次 tutorial 你答对了什么 / 答错了什么 ---
print("[TASK] 任务级反馈 (关于这次的具体任务表现):")
for topic, score in mastery.items():
    if score >= 0.6:
        print(f"  + {topic}: 覆盖率 {score:.0%}, 关键词命中达标。但不要满足, 命中关键词不等于能上生产。")
    else:
        print(f"  - {topic}: 覆盖率 {score:.0%}, 关键词未命中, 你的陈述太模糊, 不经 TLA 直接做 AT 会失败。")

# --- [PROCESS] 过程级反馈: 你是怎么思考的, 思考路径对不对 ---
print("\n[PROCESS] 过程级反馈 (关于你的推理路径):")
if any("fallback" in t for t in mastery):
    print("  你在灾备题上倾向'背顺序'而非'想异常类型'。正确路径: 先列可能异常 (RateLimitError / "
          "APIConnectionError / Timeout / Redis 不可用), 再为每种异常设计 fallback 级。")
if any("tiktoken" in t or "cl100k" in t for t in mastery):
    print("  你在成本题上正确选了 cl100k_base, 但推理路径要补一步: 验证编码 (用 tiktoken.encoding_for_model('gpt-4o') "
          "查实际编码名), 不要靠记忆。")
print("  通用过程建议: 每答一题先自问'这个设计在压测 50 并发下还成立吗?', 把压测当默认验证手段。")

# --- [SELF-REG] 自我调节级反馈: 你如何监控自己的学习 ---
print("\n[SELF-REG] 自我调节反馈 (关于你如何监控自己的盲点):")
print(f"  本 tutorial 你共 {turn} 轮, 累计盲点 {len(blindspots)} 处。")
print("  你需要建立自检清单: 每次提交前问自己 (1) 我用了 tiktoken 还是 len 估算? "
          "(2) 我的 fallback 各级捕获的是特定异常还是裸 except? "
          "(3) 我的 CI 门禁有 3 个阈值吗 (完成率/幻觉率/安全违规率)?")
print("  自检清单不过关, 不要提交 AT 评估任务。")

# --- [FEED-FORWARD] 前馈反馈: 下一步该做什么 ---
print("\n[FEED-FORWARD] 前馈反馈 (关于下一步去哪):")
weak = [t for t, s in mastery.items() if s < 0.6]
if weak:
    print(f"  你的弱项主题: {weak}")
    print("  下一步: 进入 practice.md Weak Loop -> 回退 Faded -> 补 Worked -> 复述决策点 -> 重做 Independent。")
    print("  复习单元: 重读 notes.md 关键回顾 + starter.ipynb 对应 TODO + reading.md 对应深链。")
else:
    print("  全部达标。下一步: 进入 alignment.md 对应 AT (Drill Independent 阶段), "
          "并用 schedule.json C1-C8 卡片做 FSRS-6 间隔重复防遗忘。")
    print("  推荐复习单元: Day 3 (deepeval 测试套件可嵌入 CI 门禁) + Day 4 (安全违规率=0% 的来源)。")

# 防 Self 级表扬: 不输出"你真棒/做得好"等无信息反馈
print("\n(本反馈遵循 Hattie 原则: 无 Self 级表扬, 聚焦 Task/Process/Self-Reg/Feed-Forward。)")


## Cell 6 · 限频 + Exit Artifact

### 限频 (防依赖)

- **每单元每天 1 次 tutorial**: 本 notebook 每个 calendar day 最多跑 1 次完整 4 轮 Socratic loop。原因: 牛津 tutorial 的价值在于 pre-tutorial retrieval 的强制提取, 频繁跑会让学生跳过 retrieval 直接试答案, 丧失学习效果 (Karpicke & Roediger 2008 retrieval practice effect)。
- **超频处理**: 若 student_model.json 的 `completed_rounds` 显示当日已跑过, Cell 3 应直接退出并提示"今日已用完 1 次额度, 明日再来, 期间用 schedule.json 卡片做间隔重复"。
- **防 API 依赖**: 本 notebook 全程静态 if/else, 不调真实 LLM API, 也不应被改成调 API (会破坏限频机制 + 违反 anti-stall)。

### Exit Artifact (tutorial 结束必交)

tutorial 跑完后, 在自己的学习日志里写以下 3 项 (交给导师或贴 alignment.md AT 评论区):

1. **2-3 个盲点** (从 Cell 4 student_model.json 的 `blindspots` 取): 具体到"漏了哪些关键词 / 哪个推理路径错了", 不要写"我整体不错"。
2. **推荐复习单元**: 基于盲点, 指明回看哪些 notes.md 节 / starter.ipynb TODO / reading.md 深链 / 哪个 Day 的内容 (Day 3 deepeval / Day 4 安全)。
3. **下次 tutorial 的 pre-task 草稿提纲** (>=150 字): 针对盲点写一段新的 retrieval, 下次 tutorial 前完成。

> Exit Artifact 未交 = 本单元 mastery 未达, 触发 alignment.md 的 Feed Forward 自检 (不经 TLA 能过 AT 吗? 若能 = 对齐失败)。